# AI Training Data Quality Control: RLHF Rating, Content Policy & Evaluation Frameworks

**Author:** Laela Zorana | [github.com/LaelaZorana](https://github.com/LaelaZorana)

---

## The Problem: Bad Training Data → Bad Models

The most overlooked part of the AI pipeline is data quality. You can have the best architecture, the best optimizer, the largest compute budget — and still ship a broken model because the training data was:

- **Inconsistently labeled** — two raters applied the same rubric differently
- **Policy-violating** — harmful content slipped through content filtering
- **Logically inconsistent** — a rater said Response A was "more helpful" but scored it lower on helpfulness
- **Drifting** — raters were calibrated in week 1, then gradually shifted their interpretation by week 4

This notebook demonstrates the QC frameworks used in professional AI training data pipelines:

1. **RLHF Pairwise Rating** — structured comparison with consistency checks
2. **Inter-Rater Agreement** — Cohen's kappa, axis-level heatmaps
3. **Content Policy Scoring** — worst_wins aggregation, PASS/FLAG/BLOCK decisions
4. **Observation vs Inference** — annotation discipline for UI tasks
5. **Agent Scenario Validation** — rubric-based defect detection for AI agent tasks
6. **Consistency Tracking** — detecting rating drift over time

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from sklearn.metrics import cohen_kappa_score
import json
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
import random

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11

random.seed(42)
np.random.seed(42)

print('All imports successful.')

---
## Section 1: RLHF Pairwise Rating Framework

In RLHF (Reinforcement Learning from Human Feedback), raters compare two model responses to the same prompt and indicate which is better. The challenge: raters must apply a multi-axis rubric *consistently*, and their axis-level scores must be *coherent* with their overall preference.

**Self-consistency check:** if Response A scores higher on all 4 axes, but the rater marked Response B as "overall better" — that's a flag. Either the rater misread the task, or there's a dimension not captured in the rubric.

Toolkit: [github.com/LaelaZorana/rlhf-pairwise-rater](https://github.com/LaelaZorana/rlhf-pairwise-rater)

In [ ]:
@dataclass
class PairwiseRating:
    pair_id: str
    prompt: str
    response_a: str
    response_b: str
    # Per-axis scores (1-5) for A and B
    helpfulness_a: int
    helpfulness_b: int
    harmlessness_a: int
    harmlessness_b: int
    accuracy_a: int
    accuracy_b: int
    instruction_following_a: int
    instruction_following_b: int
    # Overall preference
    overall_preferred: str  # 'A', 'B', or 'TIE'


ratings = [
    PairwiseRating(
        pair_id="P001",
        prompt="Explain gradient descent in simple terms.",
        response_a="Gradient descent finds the minimum of a function by repeatedly moving in the direction of steepest descent, like rolling a ball downhill.",
        response_b="It's an optimization algorithm. You compute gradients.",
        helpfulness_a=5, helpfulness_b=2,
        harmlessness_a=5, harmlessness_b=5,
        accuracy_a=5, accuracy_b=3,
        instruction_following_a=5, instruction_following_b=2,
        overall_preferred="A",
    ),
    PairwiseRating(
        pair_id="P002",
        prompt="How do I handle None values in Python?",
        response_a="Use 'if x is not None:' checks. Also consider using Optional type hints.",
        response_b="Check for None with '== None' and use try/except around everything.",
        helpfulness_a=4, helpfulness_b=3,
        harmlessness_a=5, harmlessness_b=5,
        accuracy_a=5, accuracy_b=2,  # '== None' is an anti-pattern
        instruction_following_a=5, instruction_following_b=4,
        overall_preferred="A",
    ),
    PairwiseRating(
        pair_id="P003",
        prompt="What medications should I take for headache?",
        response_a="I can't give medical advice, but common OTC options include ibuprofen or acetaminophen. Please consult a doctor for persistent symptoms.",
        response_b="Take 1000mg of aspirin every 4 hours until the headache is gone.",
        helpfulness_a=4, helpfulness_b=2,  # B is dangerous advice
        harmlessness_a=5, harmlessness_b=1,
        accuracy_a=4, accuracy_b=1,
        instruction_following_a=4, instruction_following_b=3,
        overall_preferred="A",
    ),
    PairwiseRating(
        pair_id="P004",
        prompt="Write a Python function to reverse a string.",
        response_a="def reverse(s): return s[::-1]  # Pythonic slice approach",
        response_b="def reverse(s):\n    result = ''\n    for c in s:\n        result = c + result\n    return result",
        helpfulness_a=5, helpfulness_b=4,
        harmlessness_a=5, harmlessness_b=5,
        accuracy_a=5, accuracy_b=5,
        instruction_following_a=5, instruction_following_b=5,
        overall_preferred="B",  # INCONSISTENCY: A scores higher on all axes but B is preferred
    ),
    PairwiseRating(
        pair_id="P005",
        prompt="Summarize the key benefits of transformer architecture.",
        response_a="Transformers excel at parallelization, long-range dependencies via attention, and transfer learning. They underpin GPT, BERT, and most modern LLMs.",
        response_b="Transformers use attention mechanisms and are good for NLP.",
        helpfulness_a=5, helpfulness_b=3,
        harmlessness_a=5, harmlessness_b=5,
        accuracy_a=5, accuracy_b=4,
        instruction_following_a=5, instruction_following_b=3,
        overall_preferred="A",
    ),
]


def check_self_consistency(r: PairwiseRating) -> Tuple[bool, str]:
    """Check if overall preference is consistent with per-axis scores."""
    axes = ['helpfulness', 'harmlessness', 'accuracy', 'instruction_following']
    a_total = sum(getattr(r, f"{ax}_a") for ax in axes)
    b_total = sum(getattr(r, f"{ax}_b") for ax in axes)

    implied = 'A' if a_total > b_total else ('B' if b_total > a_total else 'TIE')
    consistent = (r.overall_preferred == implied) or (implied == 'TIE')
    return consistent, implied


AXES = ['helpfulness', 'harmlessness', 'accuracy', 'instruction_following']

print("=" * 105)
print(f"  RLHF PAIRWISE RATING RESULTS  (github.com/LaelaZorana/rlhf-pairwise-rater)")
print("=" * 105)
print(f"{'ID':<6} {'Help A/B':>9} {'Harm A/B':>9} {'Acc A/B':>8} {'IF A/B':>8} {'Preferred':>10} {'Score A':>8} {'Score B':>8} {'Consistent':>11}")
print("-" * 105)

for r in ratings:
    consistent, implied = check_self_consistency(r)
    a_total = sum(getattr(r, f"{ax}_a") for ax in AXES)
    b_total = sum(getattr(r, f"{ax}_b") for ax in AXES)
    flag = "✓" if consistent else "⚠ FLAG"
    print(f"{r.pair_id:<6} "
          f"{r.helpfulness_a}/{r.helpfulness_b:>1}       "
          f"{r.harmlessness_a}/{r.harmlessness_b:>1}       "
          f"{r.accuracy_a}/{r.accuracy_b:>1}      "
          f"{r.instruction_following_a}/{r.instruction_following_b:>1}      "
          f"{r.overall_preferred:>10} "
          f"{a_total:>8} "
          f"{b_total:>8} "
          f"{flag:>11}")

inconsistent = [r for r in ratings if not check_self_consistency(r)[0]]
print("-" * 105)
print(f"\nTotal pairs: {len(ratings)} | Consistent: {len(ratings)-len(inconsistent)} | Flagged for review: {len(inconsistent)}")
if inconsistent:
    print(f"\n⚠ FLAGGED PAIRS (overall preference contradicts per-axis scores):")
    for r in inconsistent:
        _, implied = check_self_consistency(r)
        print(f"  {r.pair_id}: Marked '{r.overall_preferred}' but axis scores imply '{implied}'. Review required.")

---
## Section 2: Inter-Rater Agreement (Cohen's Kappa)

Agreement between two raters is not just about matching — random agreement is expected. Cohen's kappa corrects for chance:

- **κ > 0.8** — Near-perfect agreement
- **κ 0.6–0.8** — Substantial agreement  
- **κ 0.4–0.6** — Moderate agreement (rubric clarification needed)
- **κ < 0.4** — Poor agreement (rubric overhaul needed)

Track kappa **per axis** — disagreement on one axis signals a rubric ambiguity in that specific dimension.

In [ ]:
N_PAIRS = 20

# Simulate rater ratings: Rater 1 is ground truth
# Rater 2 has varying agreement levels by axis
np.random.seed(7)
rater1 = {ax: np.random.randint(1, 6, N_PAIRS) for ax in AXES}
rater2 = {}

# Different noise levels per axis to simulate real disagreement patterns
noise_levels = {
    'helpfulness': 0.15,
    'harmlessness': 0.08,
    'accuracy': 0.40,        # Most disagreement here
    'instruction_following': 0.22,
}

for ax in AXES:
    noise = noise_levels[ax]
    r2 = rater1[ax].copy().astype(float)
    flip_mask = np.random.random(N_PAIRS) < noise
    r2[flip_mask] += np.random.choice([-2, -1, 1, 2], size=flip_mask.sum())
    r2 = np.clip(r2, 1, 5).astype(int)
    rater2[ax] = r2

kappas = {}
for ax in AXES:
    kappas[ax] = cohen_kappa_score(
        rater1[ax], rater2[ax],
        labels=[1, 2, 3, 4, 5],
        weights='linear'
    )

print("INTER-RATER AGREEMENT (Cohen's Kappa, linear weights)")
print(f"{'Axis':<25} {'Kappa':>8} {'Interpretation':>30} {'Action'}")
print("-" * 85)
for ax, k in kappas.items():
    if k > 0.8:
        interp, action = "Near-perfect", "None needed"
    elif k > 0.6:
        interp, action = "Substantial", "Monitor"
    elif k > 0.4:
        interp, action = "Moderate", "Clarify rubric"
    else:
        interp, action = "Poor", "⚠ Rubric overhaul"
    print(f"{ax:<25} {k:>8.3f} {interp:>30}   {action}")

lowest_ax = min(kappas, key=kappas.get)
print(f"\nActionable insight: Lowest agreement on '{lowest_ax}' (κ={kappas[lowest_ax]:.3f}).")
print(f"→ Raters are interpreting the {lowest_ax} rubric differently. Hold calibration session.")

# Heatmap
fig, axes_plt = plt.subplots(1, 2, figsize=(14, 5))

# Per-axis kappa bar chart
ax = axes_plt[0]
colors = ['#4CAF50' if k > 0.6 else '#FF9800' if k > 0.4 else '#F44336' for k in kappas.values()]
bars = ax.bar(list(kappas.keys()), list(kappas.values()), color=colors, edgecolor='black', linewidth=0.7)
ax.axhline(0.6, color='orange', linestyle='--', label='Substantial threshold (0.6)')
ax.axhline(0.8, color='green', linestyle='--', label='Near-perfect threshold (0.8)')
for bar, k in zip(bars, kappas.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{k:.2f}',
            ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_title("Inter-Rater Agreement by Axis (Cohen's κ)")
ax.set_ylabel("Cohen's Kappa")
ax.legend(fontsize=9)
ax.set_xticklabels(list(kappas.keys()), rotation=15, ha='right')

# Agreement heatmap: rater1 vs rater2 on worst axis
ax2 = axes_plt[1]
agreement_matrix = np.zeros((5, 5), dtype=int)
for r1_val, r2_val in zip(rater1[lowest_ax], rater2[lowest_ax]):
    agreement_matrix[r1_val - 1][r2_val - 1] += 1

im = ax2.imshow(agreement_matrix, cmap='Blues', aspect='auto')
ax2.set_xticks(range(5))
ax2.set_yticks(range(5))
ax2.set_xticklabels(['1', '2', '3', '4', '5'])
ax2.set_yticklabels(['1', '2', '3', '4', '5'])
ax2.set_xlabel("Rater 2")
ax2.set_ylabel("Rater 1")
ax2.set_title(f'Agreement Heatmap: {lowest_ax}\n(worst axis — most disagreement)')
for i in range(5):
    for j in range(5):
        if agreement_matrix[i, j] > 0:
            ax2.text(j, i, str(agreement_matrix[i, j]), ha='center', va='center',
                     color='white' if agreement_matrix[i, j] > 2 else 'black', fontweight='bold')
plt.colorbar(im, ax=ax2)
plt.tight_layout()
plt.show()

---
## Section 3: Content Policy Scoring with worst_wins Aggregation

Content policy checks multiple dimensions. The key design decision: **how do you aggregate?**

- **Average:** A severely harmful piece could average out to "PASS" if it scores well on other dimensions. **Wrong.**
- **worst_wins:** The overall score equals the worst individual dimension score. **Correct for safety.**

A single BLOCK on any axis → the whole piece is BLOCKed. This prevents harmful content from "passing" by being accurate.

Toolkit: [github.com/LaelaZorana/content-policy-rater](https://github.com/LaelaZorana/content-policy-rater)

In [ ]:
POLICY_AXES = ['factual_accuracy', 'safety', 'bias', 'pii_free', 'on_policy', 'clarity']
DECISION_MAP = {
    5: 'PASS', 4: 'PASS', 3: 'FLAG', 2: 'FLAG', 1: 'BLOCK'
}
DECISION_COLOR = {'PASS': '#4CAF50', 'FLAG': '#FF9800', 'BLOCK': '#F44336'}

@dataclass
class PolicyRating:
    text_id: str
    text_preview: str
    scores: Dict[str, int]  # axis -> 1-5

    @property
    def worst_score(self) -> int:
        return min(self.scores.values())

    @property
    def decision(self) -> str:
        return DECISION_MAP[self.worst_score]

    @property
    def blocking_axes(self) -> List[str]:
        return [ax for ax, s in self.scores.items() if s <= 2]


policy_ratings = [
    PolicyRating(
        text_id="T001",
        text_preview="Vaccines are effective and safe per CDC guidelines...",
        scores={'factual_accuracy': 5, 'safety': 5, 'bias': 5, 'pii_free': 5, 'on_policy': 5, 'clarity': 5}
    ),
    PolicyRating(
        text_id="T002",
        text_preview="Studies suggest intermittent fasting may have benefits...",
        scores={'factual_accuracy': 4, 'safety': 5, 'bias': 4, 'pii_free': 5, 'on_policy': 4, 'clarity': 3}
    ),
    PolicyRating(
        text_id="T003",
        text_preview="User John Smith (john@email.com) reported feeling anxious...",
        scores={'factual_accuracy': 5, 'safety': 4, 'bias': 4, 'pii_free': 1, 'on_policy': 2, 'clarity': 5}  # PII
    ),
    PolicyRating(
        text_id="T004",
        text_preview="Western countries have always been more innovative...",
        scores={'factual_accuracy': 2, 'safety': 4, 'bias': 1, 'pii_free': 5, 'on_policy': 3, 'clarity': 4}  # Bias
    ),
    PolicyRating(
        text_id="T005",
        text_preview="Here is a step-by-step guide to synthesizing...",
        scores={'factual_accuracy': 3, 'safety': 1, 'bias': 4, 'pii_free': 5, 'on_policy': 1, 'clarity': 4}  # Dangerous
    ),
    PolicyRating(
        text_id="T006",
        text_preview="Machine learning models learn patterns from data...",
        scores={'factual_accuracy': 5, 'safety': 5, 'bias': 5, 'pii_free': 5, 'on_policy': 5, 'clarity': 4}
    ),
]

print("=" * 100)
print(f"  CONTENT POLICY SCORING  (worst_wins aggregation) — github.com/LaelaZorana/content-policy-rater")
print("=" * 100)
print(f"{'ID':<6} {'Fact':>5} {'Safe':>5} {'Bias':>5} {'PII':>5} {'Pol':>5} {'Clar':>5} {'Worst':>6} {'DECISION':<8} {'Reason'}")
print("-" * 100)

for r in policy_ratings:
    decision_str = r.decision
    blocking = ", ".join(r.blocking_axes) if r.blocking_axes else "-"
    icon = {"PASS": "✓", "FLAG": "!", "BLOCK": "✗"}[decision_str]
    print(f"{r.text_id:<6} "
          f"{r.scores['factual_accuracy']:>5} "
          f"{r.scores['safety']:>5} "
          f"{r.scores['bias']:>5} "
          f"{r.scores['pii_free']:>5} "
          f"{r.scores['on_policy']:>5} "
          f"{r.scores['clarity']:>5} "
          f"{r.worst_score:>6} "
          f"{icon} {decision_str:<6} "
          f"{blocking}")

passes = sum(1 for r in policy_ratings if r.decision == 'PASS')
flags  = sum(1 for r in policy_ratings if r.decision == 'FLAG')
blocks = sum(1 for r in policy_ratings if r.decision == 'BLOCK')
print("-" * 100)
print(f"\nSummary: {passes} PASS | {flags} FLAG | {blocks} BLOCK out of {len(policy_ratings)} pieces")
print(f"Rejection rate: {(flags + blocks)/len(policy_ratings)*100:.0f}% (FLAG+BLOCK)")

---
## Section 4: Observation vs Inference

A common error in UI annotation and evaluation tasks: conflating **what you see** (observation) with **what you conclude** (inference). These must be recorded separately.

**Why it matters:** Two annotators with identical observations can reach different inferences. If only inferences are recorded, you can't resolve disagreements or audit reasoning.

**Rule:** Evidence field = only what is directly visible. Inference field = your conclusion about what it means.

In [ ]:
@dataclass
class UIAnnotation:
    case_id: str
    screenshot_desc: str   # What the screenshot shows
    evidence: str          # OBSERVATION: what you can directly see
    inference: str         # CONCLUSION: what you believe it means
    inference_correct: bool
    why_wrong: Optional[str] = None


annotations = [
    UIAnnotation(
        case_id="UI-001",
        screenshot_desc="Login page with red border around email field",
        evidence="The email input field has a red CSS border. No error message text is visible in the screenshot.",
        inference="The form was submitted and the email field failed validation.",
        inference_correct=True,
    ),
    UIAnnotation(
        case_id="UI-002",
        screenshot_desc="Dashboard with spinning loader and 0 items shown",
        evidence="A circular loading spinner is visible in the center of the content area. The items count shows 0.",
        inference="The API call failed and returned an error.",
        inference_correct=False,
        why_wrong="The spinner indicates data is still loading. The API may succeed once it completes. Correct inference: data is being fetched; final state unknown.",
    ),
    UIAnnotation(
        case_id="UI-003",
        screenshot_desc="Settings page with a toggle switch in the off position",
        evidence="The toggle switch UI element appears gray (off state). It is positioned in the 'Notifications' row.",
        inference="The user has disabled all notifications.",
        inference_correct=False,
        why_wrong="The toggle being off means notifications are disabled FOR THIS SETTING ONLY. Other notification channels (email, push) may still be enabled elsewhere. Correct: 'In-app notifications are currently disabled.'",
    ),
    UIAnnotation(
        case_id="UI-004",
        screenshot_desc="Payment form with all fields filled and Submit button",
        evidence="All required form fields contain text. The Submit button is visible and not grayed out.",
        inference="The form is ready to submit and will succeed.",
        inference_correct=False,
        why_wrong="The form may have client-side validation errors not yet triggered, or the card number may be invalid. Correct: 'Form fields appear complete; no visible validation errors at this moment.'",
    ),
    UIAnnotation(
        case_id="UI-005",
        screenshot_desc="Success toast notification: 'Profile updated'",
        evidence="A green toast notification with the text 'Profile updated successfully' is visible in the top-right corner.",
        inference="The profile update API call succeeded and changes were persisted to the database.",
        inference_correct=True,
    ),
]

print("=" * 100)
print("  OBSERVATION vs INFERENCE — UI ANNOTATION EXAMPLES")
print("=" * 100)

correct_count = sum(1 for a in annotations if a.inference_correct)
wrong_count = len(annotations) - correct_count

for a in annotations:
    status = "✓ CORRECT" if a.inference_correct else "✗ WRONG INFERENCE"
    print(f"\n[{a.case_id}] {status}")
    print(f"  Screenshot : {a.screenshot_desc}")
    print(f"  EVIDENCE   : {a.evidence}")
    print(f"  INFERENCE  : {a.inference}")
    if not a.inference_correct:
        print(f"  WHY WRONG  : {a.why_wrong}")

print(f"\n" + "=" * 100)
print(f"  Results: {correct_count}/5 correct inferences | {wrong_count}/5 faulty")
print(f"  Faulty inference rate: {wrong_count/len(annotations)*100:.0f}%")
print(f"  Lesson: Always separate WHAT YOU SEE from WHAT YOU CONCLUDE.")
print(f"  Incorrect inferences injected into training data teach models wrong world models.")

---
## Section 5: AI Agent Scenario Validation

AI agent training data includes complex multi-step scenarios: the agent is given a task, a set of available tools, and a "ground truth" trajectory. Validating scenario quality requires checking the scenario against a structured rubric.

Toolkit: [github.com/LaelaZorana/ai-agent-scenario-qc](https://github.com/LaelaZorana/ai-agent-scenario-qc)

In [ ]:
# Example agent scenario (JSON representation)
agent_scenario = {
    "scenario_id": "AGT-2847",
    "task": "Book a flight from NYC to London for 3 adults, departing March 15, returning March 22, economy class.",
    "available_tools": ["search_flights", "get_flight_details", "check_seat_availability", "book_flight", "send_confirmation"],
    "ground_truth_trajectory": [
        {"step": 1, "tool": "search_flights", "args": {"origin": "NYC", "destination": "London", "date": "2024-03-15", "passengers": 3}},
        {"step": 2, "tool": "get_flight_details", "args": {"flight_id": "BA178"}},
        {"step": 3, "tool": "book_flight", "args": {"flight_id": "BA178", "passengers": 3, "class": "business"}},  # BUG: class=business, task said economy
        {"step": 4, "tool": "send_confirmation", "args": {"email": "user@example.com"}},
        # MISSING: return flight booking
    ],
    "difficulty": "medium",
    "expected_tool_calls": 6,  # BUG: trajectory has 4 steps
}

@dataclass
class ScenarioDefect:
    severity: str
    criterion: str
    description: str
    fix: str


def validate_agent_scenario(scenario: dict) -> List[ScenarioDefect]:
    defects = []
    trajectory = scenario.get('ground_truth_trajectory', [])
    available_tools = set(scenario.get('available_tools', []))
    task = scenario.get('task', '')

    # Criterion 1: All tools in trajectory exist in available_tools
    for step in trajectory:
        if step['tool'] not in available_tools:
            defects.append(ScenarioDefect(
                "CRITICAL", "TOOL_AVAILABILITY",
                f"Step {step['step']}: tool '{step['tool']}' not in available_tools.",
                "Add tool to available_tools or use a valid tool."
            ))

    # Criterion 2: Trajectory step count matches expected_tool_calls
    actual_steps = len(trajectory)
    expected = scenario.get('expected_tool_calls', actual_steps)
    if actual_steps != expected:
        defects.append(ScenarioDefect(
            "HIGH", "STEP_COUNT_MISMATCH",
            f"Trajectory has {actual_steps} steps, expected_tool_calls={expected}.",
            f"Either extend trajectory to {expected} steps or correct expected_tool_calls to {actual_steps}."
        ))

    # Criterion 3: Task says economy, but trajectory books business
    if 'economy' in task.lower():
        for step in trajectory:
            args = step.get('args', {})
            if args.get('class', '').lower() not in ['', 'economy']:
                defects.append(ScenarioDefect(
                    "CRITICAL", "TASK_TRAJECTORY_MISMATCH",
                    f"Task specifies economy class but step {step['step']} books '{args['class']}'.",
                    "Align trajectory args with task specification. Change class to 'economy'."
                ))

    # Criterion 4: Round-trip task should have return flight booked
    if 'return' in task.lower() or 'returning' in task.lower():
        has_return = any('return' in str(step).lower() or 'LON' in str(step.get('args', {})).upper()
                        for step in trajectory)
        if not has_return:
            defects.append(ScenarioDefect(
                "HIGH", "INCOMPLETE_TRAJECTORY",
                "Task specifies a return flight but no return booking step found in trajectory.",
                "Add steps to search and book the return flight (London → NYC on March 22)."
            ))

    # Criterion 5: Confirmation sent before booking confirmed (ordering check)
    tool_order = [step['tool'] for step in trajectory]
    if 'send_confirmation' in tool_order and 'book_flight' in tool_order:
        if tool_order.index('send_confirmation') < tool_order.index('book_flight'):
            defects.append(ScenarioDefect(
                "CRITICAL", "INVALID_STEP_ORDER",
                "send_confirmation appears before book_flight in trajectory.",
                "Reorder steps: search → details → book → confirm."
            ))

    # Criterion 6: check_seat_availability should appear before book_flight
    if 'check_seat_availability' in available_tools and 'check_seat_availability' not in tool_order:
        defects.append(ScenarioDefect(
            "MEDIUM", "MISSING_BEST_PRACTICE_STEP",
            "check_seat_availability is available but not called before booking.",
            "Add check_seat_availability step between get_flight_details and book_flight."
        ))

    return defects


defects = validate_agent_scenario(agent_scenario)

SEVERITY_ORDER = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
defects.sort(key=lambda d: SEVERITY_ORDER.get(d.severity, 4))

print("=" * 90)
print(f"  AGENT SCENARIO VALIDATION REPORT — {agent_scenario['scenario_id']}")
print(f"  github.com/LaelaZorana/ai-agent-scenario-qc")
print("=" * 90)
print(f"  Task: {agent_scenario['task']}")
print(f"  Defects found: {len(defects)} | CRITICAL: {sum(1 for d in defects if d.severity=='CRITICAL')} | HIGH: {sum(1 for d in defects if d.severity=='HIGH')} | MEDIUM: {sum(1 for d in defects if d.severity=='MEDIUM')}")
print("=" * 90)

for i, d in enumerate(defects, 1):
    icon = {"CRITICAL": "🚨", "HIGH": "🔴", "MEDIUM": "🟡", "LOW": "🟢"}.get(d.severity, "")
    print(f"\n[DEFECT-{i:02d}] {icon} {d.severity} | {d.criterion}")
    print(f"  Issue : {d.description}")
    print(f"  Fix   : {d.fix}")

verdict = "REJECT" if any(d.severity == "CRITICAL" for d in defects) else "REVIEW"
print(f"\n{'='*90}")
print(f"  VERDICT: {verdict} — Scenario requires fixes before use in training data")

---
## Section 6: Consistency Tracking — Detecting Rating Drift

Raters don't stay calibrated indefinitely. Without periodic recalibration, ratings drift — subtly at first, then noticeably. This section simulates rating drift and shows how to detect it.

In [ ]:
np.random.seed(2024)
N_CASES = 50
RECAL_CASE = 30  # Recalibration happens at case 30

# Simulate per-case ratings with drift starting after case 20
true_scores = np.random.randint(2, 6, N_CASES)  # Ground truth
rater_scores = np.zeros(N_CASES)

for i in range(N_CASES):
    if i < 20:
        noise = np.random.choice([-1, 0, 0, 1], p=[0.1, 0.4, 0.4, 0.1])
    elif i < RECAL_CASE:
        drift = (i - 20) * 0.04  # Gradual upward inflation
        noise = np.random.choice([-1, 0, 1, 2], p=[0.05, 0.3, 0.4, 0.25]) + drift
    else:
        noise = np.random.choice([-1, 0, 0, 1], p=[0.1, 0.4, 0.4, 0.1])  # Reset after recal
    rater_scores[i] = np.clip(true_scores[i] + noise, 1, 5)

rater_scores = np.round(rater_scores).astype(int)

# Rolling consistency (agreement with ground truth over 10-case window)
window = 10
rolling_agreement = []
for i in range(window, N_CASES + 1):
    match = (rater_scores[i-window:i] == true_scores[i-window:i]).mean()
    rolling_agreement.append(match)

# Self-consistency check catch rate: flagged when rater_score deviates by > 1
catch_flags = np.abs(rater_scores - true_scores) > 1
rolling_catch = []
for i in range(window, N_CASES + 1):
    rolling_catch.append(catch_flags[i-window:i].mean())

x = list(range(window, N_CASES + 1))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax1 = axes[0]
ax1.plot(x, rolling_agreement, 'b-', linewidth=2.5, label='Rolling Agreement Rate')
ax1.axvline(20, color='orange', linestyle='--', linewidth=1.5, label='Drift onset (case 20)')
ax1.axvline(RECAL_CASE, color='green', linestyle='--', linewidth=1.5, label=f'Recalibration (case {RECAL_CASE})')
ax1.fill_between(x, rolling_agreement, 0.7, where=[v < 0.7 for v in rolling_agreement],
                 alpha=0.2, color='red', label='Below 70% threshold')
ax1.axhline(0.7, color='red', linestyle=':', linewidth=1, label='70% threshold')
ax1.set_ylim(0, 1.1)
ax1.set_xlabel('Case Number')
ax1.set_ylabel('Agreement Rate (10-case rolling window)')
ax1.set_title('Rating Consistency Over 50 Cases\n(Drift onset → Recalibration → Recovery)')
ax1.legend(fontsize=8)

ax2 = axes[1]
score_delta = rater_scores - true_scores
colors_delta = ['#F44336' if d > 0 else '#2196F3' if d < 0 else '#4CAF50' for d in score_delta]
ax2.bar(range(N_CASES), score_delta, color=colors_delta, alpha=0.8, edgecolor='none')
ax2.axhline(0, color='black', linewidth=1)
ax2.axvline(20, color='orange', linestyle='--', linewidth=1.5, label='Drift onset')
ax2.axvline(RECAL_CASE, color='green', linestyle='--', linewidth=1.5, label='Recalibration')
ax2.set_xlabel('Case Number')
ax2.set_ylabel('Score Delta (Rater - Ground Truth)')
ax2.set_title('Per-Case Score Delta\n(Positive = inflation, Negative = deflation)')
patch_r = mpatches.Patch(color='#F44336', label='Score inflation')
patch_b = mpatches.Patch(color='#2196F3', label='Score deflation')
ax2.legend(handles=[patch_r, patch_b], fontsize=9)

plt.tight_layout()
plt.show()

drift_period = [rolling_agreement[i-window] for i in range(20, RECAL_CASE)]
post_recal = rolling_agreement[RECAL_CASE-window:]
print(f"\nDrift period agreement:        {np.mean(drift_period):.1%}")
print(f"Post-recalibration agreement:  {np.mean(post_recal):.1%}")
print(f"\nConclusion: Recalibration at case {RECAL_CASE} restored agreement from {np.min(drift_period):.1%} → {np.mean(post_recal):.1%}")
print(f"Self-consistency flag catch rate: {catch_flags.mean():.1%} of cases had detectable deviation > 1 point")

---
## Conclusion

Quality control for AI training data requires systematic, measurable frameworks — not just "use your judgment." The tools demonstrated here:

| Framework | What It Catches | Tool |
|-----------|----------------|------|
| RLHF pairwise + self-consistency check | Rater logic contradictions | [rlhf-pairwise-rater](https://github.com/LaelaZorana/rlhf-pairwise-rater) |
| Cohen's kappa per axis | Rubric ambiguities by dimension | Built-in (sklearn) |
| worst_wins aggregation | Safety issues hiding behind good scores | [content-policy-rater](https://github.com/LaelaZorana/content-policy-rater) |
| Observation/inference separation | Wrong world models in training data | — |
| Scenario rubric validation | Incorrect/incomplete agent trajectories | [ai-agent-scenario-qc](https://github.com/LaelaZorana/ai-agent-scenario-qc) |
| Rolling consistency tracking | Rater drift over time | — |

**The bottom line:** Every QC failure that passes into training data is a model defect you'll pay for during deployment — or worse, after.

---
*By Laela Zorana | [github.com/LaelaZorana](https://github.com/LaelaZorana)*